# Notebook 2 — Preprocessing

This notebook transforms the raw FER2013 arrays produced by the data loader into
training-ready tensors. We produce **two versions** of the processed data:

- **Grayscale 48 × 48 × 1** — native FER2013 resolution, used by the custom CNN.
- **RGB 64 × 64 × 3** — upscaled and channel-duplicated, used by MobileNetV2.

Both versions are persisted to `data/processed/` so that the training notebook
(Notebook 3) can reload them in seconds without repeating this pipeline.
We also compute class weights to handle the significant class imbalance in
FER2013, and configure and visualise the data-augmentation strategy.

## Section 1 — Setup and data loading

We fix all random seeds before any array manipulation so that the train/val split
is reproducible across runs. The raw images are loaded as uint8 grayscale arrays
of shape `(n, 48, 48)` via `load_fer2013`, which also handles the stratified
validation split. Printing shapes here confirms that we start from the expected
data before any transformation.

In [ ]:
import random
import sys
from pathlib import Path

import numpy as np
import tensorflow as tf

# Make src importable from the notebook directory
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.data.loader import load_fer2013

SEED = config.RANDOM_SEED
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

X_train, y_train, X_val, y_val, X_test, y_test = load_fer2013(config.RAW_DIR)

print(f"X_train: {X_train.shape}  dtype={X_train.dtype}")
print(f"X_val:   {X_val.shape}  dtype={X_val.dtype}")
print(f"X_test:  {X_test.shape}  dtype={X_test.dtype}")
print(f"y_train: {y_train.shape}  labels {y_train.min()}–{y_train.max()}")

## Section 2 — Normalization

Pixel values are divided by 255 to map from uint8 [0, 255] to float32 [0, 1].
Keeping raw byte values would cause very large activations in the first layer,
slowing convergence and making gradient-based optimisers less stable.
MobileNetV2 additionally expects inputs in this range because its batch-norm
statistics were calibrated during ImageNet pretraining on [0, 1] tensors.
Decision: simple division by 255 rather than per-channel standardisation
(mean/std), because FER2013 is grayscale and the channel-specific statistics
would add complexity without meaningful benefit for this dataset.

In [ ]:
X_train_f = X_train.astype(np.float32) / 255.0
X_val_f   = X_val.astype(np.float32)   / 255.0
X_test_f  = X_test.astype(np.float32)  / 255.0

print(f"Pixel range after normalisation: [{X_train_f.min():.3f}, {X_train_f.max():.3f}]")
print(f"dtype: {X_train_f.dtype}")

## Section 3 — Reshape for the custom CNN

Keras `Conv2D` layers expect a 4-D input `(batch, height, width, channels)`.
The loader returns 3-D arrays `(n, 48, 48)`, so we add a trailing axis of size 1
to produce `(n, 48, 48, 1)`. This represents a single grayscale channel and is
the correct input shape for our custom CNN.

In [ ]:
X_train_gray = X_train_f[..., np.newaxis]   # (n, 48, 48, 1)
X_val_gray   = X_val_f[..., np.newaxis]
X_test_gray  = X_test_f[..., np.newaxis]

print(f"Custom CNN input — train: {X_train_gray.shape}, val: {X_val_gray.shape}, test: {X_test_gray.shape}")